<a href="https://colab.research.google.com/github/balu-karthik/document-processing/blob/main/LLM_Tools_Clean.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LLM Tools: Tiktoken, LiteLLM and Langfuse

1. **Tiktoken** - count tokens and estimate cost
2. **LiteLLM** - call a model, read token usage and cost
3. **Langfuse** - trace every call, group sessions and enforce a budget

## Tool 1: Tiktoken

In [ ]:
!pip install -q tiktoken

### 1. Count tokens

In [ ]:
import tiktoken

prompt = """
Explain how machine learning can be used
for stress detection using wearable sensors.
"""

encoding = tiktoken.get_encoding("o200k_base")
tokens = encoding.encode(prompt)

print("Tokens:", tokens)
print("Number of tokens:", len(tokens))

Tokens: [198, 176289, 1495, 7342, 7524, 665, 413, 2061, 198, 1938, 9260, 28385, 2360, 122441, 39717, 558]
Number of tokens: 16


### 2. Estimate cost and check budget

In [ ]:
# User budget
budget = 10.00

# Example model pricing (USD per million tokens)
input_price_per_million = 0.20
output_price_per_million = 1.20

# Input tokens
input_tokens = len(tokens)

# Estimated output
estimated_output_tokens = 500

# Cost calculation
input_cost = (input_tokens / 1_000_000) * input_price_per_million
output_cost = (estimated_output_tokens / 1_000_000) * output_price_per_million
total_cost = input_cost + output_cost

print("Input tokens:", input_tokens)
print("Estimated output tokens:", estimated_output_tokens)
print("Input cost: $", input_cost)
print("Output cost: $", output_cost)
print("Estimated total cost: $", total_cost)

if total_cost <= budget:
    print("Query is affordable")
    print("Remaining budget: $", budget - total_cost)
else:
    print("Query exceeds budget")

Input tokens: 16
Estimated output tokens: 500
Input cost: $ 3.2e-06
Output cost: $ 0.0006
Estimated total cost: $ 0.0006031999999999999
Query is affordable
Remaining budget: $ 9.9993968


## Tool 2: LiteLLM

In [ ]:
!pip install -q -U litellm anthropic

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.9/36.9 MB 37.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 49.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.3/278.3 kB 19.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.2/100.2 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.0/16.0 MB 80.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 kB 7.0 MB/s eta 0:00:00


### 1. Load the Anthropic API key securely from Colab Secrets

In [ ]:
from google.colab import userdata
import os

os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")

print("Anthropic API key loaded from Colab Secrets.")

Anthropic API key loaded from Colab Secrets.


### 2. Make one Claude request using LiteLLM

In [ ]:
import litellm
from litellm import completion

# Send traces to Langfuse (via OpenTelemetry)
litellm.callbacks = ["langfuse_otel"]

response = completion(
    model="anthropic/claude-sonnet-5-5",
    messages=[
        {
            "role": "user",
            "content": "Explain me a binary search tree in five lines"
        }
    ],
    max_tokens=200
)

print("ANSWER:")
print(response.choices[0].message.content)

11:18:32 - LiteLLM:WARNING: opentelemetry.py:487 - Proxy Server is not installed. Skipping OpenTelemetry initialization.


ANSWER:
A **binary search tree (BST)** is a tree data structure where each node has at most two children: left and right.

1. **Ordering rule:** For every node, all values in its left subtree are smaller, and all values in its right subtree are larger.
2. **Search:** Start at the root; go left if your target is smaller, right if it's larger, until you find it or hit an empty spot.
3. **Insert/Delete:** Both follow the same path as search, then add or remove the node (deletion may need rearranging children).
4. **Speed:** Operations take O(log n) time on average, but degrade to O(n) if the tree becomes lop


### 3. Show token usage and calculate actual cost

In [ ]:
import litellm

input_tokens = response.usage.prompt_tokens
output_tokens = response.usage.completion_tokens
total_tokens = response.usage.total_tokens

cost = litellm.completion_cost(
    completion_response=response
)

print("\nUSAGE:")
print("Input tokens :", input_tokens)
print("Output tokens:", output_tokens)
print("Total tokens :", total_tokens)

print("\nCOST:")
print(f"${cost:.8f}")


USAGE:
Input tokens : 22
Output tokens: 200
Total tokens : 222

COST:
$0.00204400


### 4. Check the actual request against a budget

In [ ]:
budget = 10.00

if cost <= budget:
    print(f"Affordable. Remaining budget: ${budget - cost:.4f}")
else:
    print("Budget exceeded.")

Affordable. Remaining budget: $9.9980


### 5. Automatically track request cost with a LiteLLM callback

In [ ]:
import litellm
from litellm import completion

# Callback to track cost
def track_cost(kwargs, completion_response, start_time, end_time):
    cost = kwargs.get("response_cost", 0)
    print(f"Request cost from callback: ${cost:.8f}")

litellm.success_callback = [track_cost]


# Make the request
response = completion(
    model="anthropic/claude-sonnet-5-5",
    messages=[
        {
            "role": "user",
            "content": "Explain stress detection using physiological signals."
        }
    ],
    max_tokens=200
)

# Print response
print("\nClaude Response:")
print(response.choices[0].message.content)

# Print token usage
print("\nToken Usage:")
print("Input tokens :", response.usage.prompt_tokens)
print("Output tokens:", response.usage.completion_tokens)
print("Total tokens :", response.usage.total_tokens)


Claude Response:
# Stress Detection Using Physiological Signals

## 1. Background

When a person experiences stress, the **autonomic nervous system (ANS)** responds in two main ways:
- **Sympathetic activation** ("fight or flight"): raises heart rate, sweating, and muscle tension.
- **Parasympathetic withdrawal**: reduces the calming "rest and digest" influence.

The **HPA axis** also releases cortisol. Many of these changes can be measured with sensors, which makes objective, continuous stress monitoring possible without relying only on self-reports.

---

## 2. Key Physi

Token Usage:
Input tokens : 22
Output tokens: 200
Total tokens : 222


## Tool 3: Langfuse

In [ ]:
!pip install -q -U langfuse

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 708.2/708.2 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.3/60.3 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.3/206.3 kB 11.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.1 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.1 which is incompatible.


### 1. Load Langfuse credentials from Colab Secrets

In [ ]:
from google.colab import userdata
import os

os.environ["LANGFUSE_PUBLIC_KEY"] = userdata.get("LANGFUSE_PUBLIC_KEY")
os.environ["LANGFUSE_SECRET_KEY"] = userdata.get("LANGFUSE_SECRET_KEY")
os.environ["LANGFUSE_BASE_URL"] = "https://cloud.langfuse.com"

print("Langfuse credentials loaded from Colab Secrets.")

Langfuse credentials loaded from Colab Secrets.


### 2. Authenticate with Langfuse

In [ ]:
import os
from langfuse import get_client

# Safely strip any potential trailing newlines/carriage returns from the environment variables
for key in ["LANGFUSE_PUBLIC_KEY", "LANGFUSE_SECRET_KEY", "LANGFUSE_BASE_URL"]:
    if os.environ.get(key):
        os.environ[key] = os.environ[key].strip()

langfuse = get_client()

if langfuse.auth_check():
    print("Langfuse authentication successful!")
else:
    print("Langfuse authentication failed!")

Langfuse authentication successful!


### 3. Trace a function with `@observe()`

In [ ]:
import os
from google.colab import userdata
from langfuse import observe
from litellm import completion

# Ensure the Anthropic API key is loaded into the environment variables
os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")

@observe()
def admission_assistant(question):

    response = completion(
        model="anthropic/claude-sonnet-5-5",
        messages=[
            {
                "role": "user",
                "content": question
            }
        ],
        max_tokens=200
    )

    return response.choices[0].message.content


answer = admission_assistant(
    "What documents do I need to apply for college admission?"
)

print(answer)

# Documents for a College Application

Requirements vary by school and country, but these are the most common.

## Core Documents
- **Application form**, such as the Common App, Coalition App, or the school's own portal
- **Official transcripts** from your high school (and any colleges attended)
- **Standardized test scores** (SAT/ACT), though many schools are test-optional
- **Letters of recommendation**, usually 1-3 from teachers or a counselor
- **Personal essay or stat


### 4. Trace a customer-support chatbot

**Setup**

In [ ]:
import os
from anthropic import Anthropic
from langfuse import get_client

anthropic_client = Anthropic(
    api_key=os.environ["ANTHROPIC_API_KEY"]
)

langfuse = get_client()

MODEL = "claude-sonnet-5-5"

**Traced function** - each call is logged as one Langfuse *generation*

In [ ]:
def order_assistant(question):

    # Start a Langfuse generation (one model call)
    generation = langfuse.start_observation(
        name="order-support-chatbot",
        as_type="generation",
        model=MODEL,
        input=question
    )

    try:
        # Call Claude
        response = anthropic_client.messages.create(
            model=MODEL,
            max_tokens=300,
            messages=[
                {
                    "role": "user",
                    "content": question
                }
            ]
        )

        # Extract the text answer
        output = "".join(
            block.text
            for block in response.content
            if block.type == "text"
        )

        # Actual token usage
        input_tokens = response.usage.input_tokens
        output_tokens = response.usage.output_tokens

        # Send input, output and usage to Langfuse
        generation.update(
            input=question,
            output=output,
            usage_details={
                "input": input_tokens,
                "output": output_tokens
            }
        )

        # Show the result
        print("Question:", question)
        print("Answer:", output)
        print("Input tokens :", input_tokens)
        print("Output tokens:", output_tokens)
        print("Total tokens :", input_tokens + output_tokens)
        print("-" * 60)

        return output

    finally:
        # Always end the generation and send the data
        generation.end()
        langfuse.flush()

**Sample questions**

In [ ]:
queries = [
    "Where is my order?",
    "My package has not arrived yet. What should I do?",
    "My order arrived damaged. How can I get a replacement?",
    "Can I cancel my order before it is shipped?",
    "How can I get a refund for my order?",
    "Can I change my delivery address?",
    "How long does standard delivery usually take?",
    "My tracking information has not been updated for three days. What should I do?"
]

**Run all questions** - each one becomes its own trace

In [ ]:
for q in queries:
    order_assistant(q)

Question: Where is my order?
Answer: I can't see your order details or tracking information, so I can't tell you where it is. Here's how to find out:

**Check yourself**
- **Confirmation email or text:** Look for a tracking number or "Track order" link.
- **Your account:** Log in to the retailer's website or app and open "My Orders."
- **Carrier site:** If you have a tracking number, enter it on UPS, FedEx, USPS, DHL, or whichever carrier is listed.

**If it's late or stuck**
- Check for a delivery exception notice, such as a failed attempt or address problem.
- Look around your property, mailbox, and with neighbors or building staff.
- Confirm the shipping address on the order is correct.
- Contact the seller or retailer. Many will resend or refund if an order is lost.

**If you
Input tokens : 14
Output tokens: 300
Total tokens : 314
------------------------------------------------------------
Question: My package has not arrived yet. What should I do?
Answer: Here's a step-by-step ap

### 5. Add a budget guard

Stop making requests once the estimated cost reaches the budget.

In [ ]:
# Sonnet 5.5 pricing
INPUT_PRICE = 2.00 / 1_000_000
OUTPUT_PRICE = 10.00 / 1_000_000

# Your maximum budget
BUDGET = 0.01   # $0.01

In [ ]:
budget_queries = queries + [
    "Can I change my payment method?",
    "What should I do if I received the wrong product?"
]

# Running totals
total_input_tokens = 0
total_output_tokens = 0
total_cost = 0.0

# One parent observation for the whole chatbot run
chat = langfuse.start_observation(
    name="order-support-chatbot",
    as_type="span",
    input=budget_queries
)

try:

    for i, question in enumerate(budget_queries, start=1):

        # Check budget BEFORE making another API call
        if total_cost >= BUDGET:
            print("\n" + "=" * 70)
            print("BUDGET LIMIT REACHED")
            print("=" * 70)
            print(f"Queries completed : {i - 1}")
            print(f"Total input tokens : {total_input_tokens}")
            print(f"Total output tokens: {total_output_tokens}")
            print(f"Total tokens       : {total_input_tokens + total_output_tokens}")
            print(f"Total cost         : ${total_cost:.8f}")
            print(f"Budget             : ${BUDGET:.8f}")
            break

        generation = chat.start_observation(
            name="order-query",
            as_type="generation",
            model=MODEL,
            input=question
        )

        try:

            response = anthropic_client.messages.create(
                model=MODEL,
                max_tokens=300,
                messages=[
                    {
                        "role": "user",
                        "content": question
                    }
                ]
            )

            output = "".join(
                block.text
                for block in response.content
                if block.type == "text"
            )

            # Get actual token usage
            input_tokens = response.usage.input_tokens
            output_tokens = response.usage.output_tokens

            # Calculate this query's cost
            input_cost = input_tokens * INPUT_PRICE
            output_cost = output_tokens * OUTPUT_PRICE
            query_cost = input_cost + output_cost

            # Add to running totals
            total_input_tokens += input_tokens
            total_output_tokens += output_tokens
            total_cost += query_cost

            # Send usage to Langfuse
            generation.update(
                input=question,
                output=output,
                usage_details={
                    "input": input_tokens,
                    "output": output_tokens
                },
                cost_details={
                    "input": input_cost,
                    "output": output_cost,
                    "total": query_cost
                }
            )

            generation.end()

            # Display current progress
            print("\n" + "-" * 70)
            print(f"QUERY {i}")
            print("-" * 70)

            print("Question:", question)
            print("\nAnswer:", output)

            print("\nThis query:")
            print("Input tokens :", input_tokens)
            print("Output tokens:", output_tokens)
            print("Total tokens :", input_tokens + output_tokens)
            print(f"Input cost   : ${input_cost:.8f}")
            print(f"Output cost  : ${output_cost:.8f}")
            print(f"Query cost   : ${query_cost:.8f}")

            print("\nCUMULATIVE TOTAL:")
            print("Input tokens :", total_input_tokens)
            print("Output tokens:", total_output_tokens)
            print("Total tokens :", total_input_tokens + total_output_tokens)
            print(f"Total cost   : ${total_cost:.8f}")
            print(f"Budget       : ${BUDGET:.8f}")
            print(f"Remaining    : ${BUDGET - total_cost:.8f}")

        except Exception as e:
            generation.end()
            print("Error:", e)
            break

finally:

    # Store final totals in the parent observation
    chat.update(
        output={
            "queries_completed": min(i, len(budget_queries)),
            "total_input_tokens": total_input_tokens,
            "total_output_tokens": total_output_tokens,
            "total_tokens": total_input_tokens + total_output_tokens,
            "total_cost_usd": total_cost
        }
    )

    chat.end()
    langfuse.flush()


print("\n" + "=" * 70)
print("FINAL CHATBOT COST SUMMARY")
print("=" * 70)

print("Total input tokens :", total_input_tokens)
print("Total output tokens:", total_output_tokens)
print("Total tokens       :", total_input_tokens + total_output_tokens)
print(f"Total estimated cost: ${total_cost:.8f}")
print(f"Budget             : ${BUDGET:.8f}")
print(f"Remaining budget   : ${max(0, BUDGET - total_cost):.8f}")


----------------------------------------------------------------------
QUERY 1
----------------------------------------------------------------------
Question: Where is my order?

Answer: I can't see your order details or tracking information, but here's how to find it quickly.

**Check the status yourself**
1. **Confirmation email or text:** Look for a tracking number or a "Track your order" link.
2. **Your account:** Log in to the store's website or app and open "Orders" or "Order history."
3. **The carrier's website:** If you have a tracking number, enter it directly on UPS, FedEx, USPS, DHL, or whichever carrier is shipping it.

**If it's late or stuck**
- Check for a delivery attempt notice, and look around your door, mailbox, or with neighbors.
- Confirm the shipping address on the order is correct.
- Allow an extra business day or two if the tracking hasn't updated.
- If it's still missing, contact the

This query:
Input tokens : 14
Output tokens: 300
Total tokens : 314
Input c